# September Week 1 — Brownian Motion and Geometric Brownian Motion

**Project:** AI/ML-Augmented Black–Scholes Framework
**Plan reference:** §15.4 (§15.4.1–§15.4.6)
**Text:** Carol Alexander, *Market Risk Analysis Vol. I*, §I.3.7
**Asset for the model-versus-data experiment:** NIFTY 50 (`^NSEI`), 2015–2024

---

### Structure

| Part | Sections | Plan reference |
|---|---|---|
| **A — Brownian motion** | 1–11 | §15.4.1, §15.4.2 |
| **B — Geometric Brownian motion** | 12–15 | §15.4.3, §15.4.4 |
| **B — Model-versus-data experiment** | 16 | §15.4.5 |
| **Interpretation of model limitations** | 17 | §15.4.6 |

### Learning objective (§15.4.1)

$$\Delta W = W_{t+\Delta t} - W_t, \qquad \Delta W \sim N(0, \Delta t), \qquad
\Delta W = \sqrt{\Delta t}\,Z$$

leading to the asset-price model

$$\frac{dS_t}{S_t} = \mu\,dt + \sigma\,dW_t$$

### Principal question (§15.4.5)

> If GBM is calibrated to the same mean and volatility, does it reproduce the
> distributional behaviour observed in the actual market data?

Answered in Section 17 from the numerical evidence in Section 16.

---

## 0. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

SEED = 20260825                     # same seed convention as Week 1
rng  = np.random.default_rng(SEED)

plt.rcParams.update({
    "figure.figsize": (11, 4.2), "axes.grid": True, "grid.alpha": 0.3,
    "font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
})
os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)
print("NumPy", np.__version__)

---
## 1. Standard normal random variables (task 1)

Everything rests on the quality of $Z \sim N(0,1)$, so verify the generator before
building on it. With $n = 10^6$ the standard error of the sample mean is
$1/\sqrt{n} = 0.001$; the observed mean must sit within a few of those.

In [ ]:
n = 1_000_000
Z = rng.standard_normal(n)

print(f"  n            {n:,}")
print(f"  mean         {Z.mean():+.6f}   (theory 0)")
print(f"  variance     {Z.var(ddof=1):.6f}   (theory 1)")
print(f"  skewness     {stats.skew(Z):+.6f}   (theory 0)")
print(f"  ex. kurtosis {stats.kurtosis(Z):+.6f}   (theory 0)")

se = 1/np.sqrt(n)
print(f"\n  |mean| / (1/sqrt(n)) = {abs(Z.mean())/se:.3f} standard errors")
assert abs(Z.mean()) < 5*se, "generator mean is implausible"

jb, p = stats.jarque_bera(Z[:5000])
print(f"  Jarque-Bera on first 5000: p = {p:.3f}  "
      f"({'consistent with normal' if p > 0.01 else 'REJECTED'})")

---
## 2. Brownian increments (task 2)

$$\Delta W = \sqrt{\Delta t}\,Z$$

Note that the scaling is $\sqrt{\Delta t}$, **not** $\Delta t$. This is the single most
important structural fact about Brownian motion and everything in Section 4 follows from it.

In [ ]:
dt, n = 1/252, 10_000
dW = np.sqrt(dt) * rng.standard_normal(n)

print(f"  dt = {dt:.6f},  n = {n:,}")
print(f"  mean(dW)  {dW.mean():+.8f}   (theory 0)")
print(f"  var(dW)   {dW.var(ddof=1):.8f}   (theory dt      = {dt:.8f})")
print(f"  sd(dW)    {dW.std(ddof=1):.8f}   (theory sqrt(dt) = {np.sqrt(dt):.8f})")

---
## 3. Complete Brownian-motion paths (task 3)

A path is the cumulative sum of increments, started at $W_0 = 0$:

$$W_{t_k} = \sum_{i=1}^{k} \Delta W_i$$

Since $W_t \sim N(0, t)$, the spread of the paths grows as $\sqrt{t}$ — the envelope widens
like a square root, not a straight line. Eight paths are far too few to test that, so the
moment checks below use 2,000.

In [ ]:
n_paths, n_steps, T = 8, 1000, 1.0
dt = T/n_steps
dW = np.sqrt(dt) * rng.standard_normal((n_paths, n_steps))
W  = np.concatenate([np.zeros((n_paths,1)), np.cumsum(dW, axis=1)], axis=1)
t  = np.linspace(0, T, n_steps+1)
assert np.all(W[:,0] == 0), "W_0 must be 0"

n_big = 2000
big = np.concatenate([np.zeros((n_big,1)),
                      np.cumsum(np.sqrt(dt)*rng.standard_normal((n_big,n_steps)), axis=1)], axis=1)
emp_sd = big.std(axis=0, ddof=1)
WT = big[:,-1]

print(f"  {n_paths} paths plotted, {n_steps} steps, T = {T},  W_0 = 0 for all")
print(f"\n  W_T moments from {n_big:,} paths:")
print(f"    mean  {WT.mean():+.4f}   theory 0        "
      f"({abs(WT.mean())/np.sqrt(T/n_big):.2f} standard errors)")
print(f"    sd    {WT.std(ddof=1):.4f}   theory sqrt(T) = {np.sqrt(T):.4f}")
print(f"    Jarque-Bera p = {stats.jarque_bera(WT)[1]:.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
for i in range(n_paths):
    ax[0].plot(t, W[i], lw=0.9)
ax[0].axhline(0, color="k", lw=0.7)
ax[0].set(xlabel="t", ylabel="$W_t$", title=f"{n_paths} Brownian-motion paths")

ax[1].plot(t, emp_sd, lw=1.4, color="#1f4e79", label="empirical SD (2000 paths)")
ax[1].plot(t, np.sqrt(t), "r--", lw=1.6, label=r"theory: $\sqrt{t}$")
ax[1].fill_between(t, -2*np.sqrt(t), 2*np.sqrt(t), alpha=0.10, color="grey",
                   label=r"$\pm 2\sqrt{t}$ envelope")
for i in range(n_paths):
    ax[1].plot(t, W[i], lw=0.5, alpha=0.45)
ax[1].set(xlabel="t", ylabel="$W_t$", title=r"Spread grows as $\sqrt{t}$, not $t$")
ax[1].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig("figures/w02_bm_paths.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 4. Moments and dependence on $\Delta t$ (tasks 4–7)

Three claims to test across nine values of $\Delta t$ spanning five orders of magnitude:

$$E[\Delta W] = 0, \qquad \mathrm{Var}(\Delta W) = \Delta t, \qquad \mathrm{SD}(\Delta W) \propto \sqrt{\Delta t}$$

**A caution on task 4.** The raw $\overline{\Delta W}$ shrinks as $\Delta t$ falls, but that
proves nothing — the whole scale is shrinking. The scale-free test is
$\overline{\Delta W}$ divided by its own standard error $\sqrt{\Delta t / n}$, which should
stay of order 1 regardless of $\Delta t$. That is the `mean / s.e.` column.

In [ ]:
dts = [1.0, 1/2, 1/4, 1/12, 1/52, 1/252, 1/1000, 1/5000, 1/20000]
n = 200_000
rows = []
for dt_ in dts:
    x = np.sqrt(dt_) * rng.standard_normal(n)
    rows.append({
        "dt":            dt_,
        "mean(dW)":      x.mean(),
        "mean / s.e.":   x.mean()/np.sqrt(dt_/n),
        "var(dW)":       x.var(ddof=1),
        "var / dt":      x.var(ddof=1)/dt_,
        "sd(dW)":        x.std(ddof=1),
        "sd / sqrt(dt)": x.std(ddof=1)/np.sqrt(dt_),
    })
tab = pd.DataFrame(rows)
tab.to_csv("results/w02_bm_scaling.csv", index=False)
with pd.option_context("display.float_format", lambda v: f"{v:12.6f}"):
    print(tab.to_string(index=False))
print(f"\n  Task 5: max |var/dt - 1| = {np.abs(tab['var / dt']-1).max():.4f}")

### Task 7 — the scaling exponent

Rather than asserting $\mathrm{SD} \propto \sqrt{\Delta t}$, fit

$$\ln \mathrm{SD}(\Delta W) = a + b\ln \Delta t$$

and check $b = 1/2$, $a = 0$. Same approach as the $r \approx R$ scaling test in Week 1.

In [ ]:
x, y = np.log(tab["dt"].values), np.log(tab["sd(dW)"].values)
slope, intercept, rval, _, stderr = stats.linregress(x, y)
print(f"  slope b     {slope:.6f}  +/- {stderr:.6f}   (theory 0.5)")
print(f"  intercept a {intercept:+.6f}                 (theory 0)")
print(f"  R^2         {rval**2:.6f}")

fig, ax = plt.subplots(1, 3, figsize=(13.5, 4.0))
ax[0].plot(tab["dt"], tab["var(dW)"], "o-", color="#1f4e79", label="empirical")
ax[0].plot(tab["dt"], tab["dt"], "r--", lw=1.5, label=r"theory: Var $= \Delta t$")
ax[0].set(xscale="log", yscale="log", xlabel=r"$\Delta t$", ylabel=r"Var$(\Delta W)$",
          title=r"Task 5:  Var$(\Delta W) = \Delta t$")
ax[0].legend(frameon=False, fontsize=8)

ax[1].plot(np.sqrt(tab["dt"]), tab["sd(dW)"], "o-", color="#1f4e79", label="empirical")
lim = np.sqrt(tab["dt"]).max()
ax[1].plot([0, lim], [0, lim], "r--", lw=1.5, label="slope 1 through origin")
ax[1].set(xlabel=r"$\sqrt{\Delta t}$", ylabel=r"SD$(\Delta W)$",
          title=r"Task 7:  SD $\propto \sqrt{\Delta t}$")
ax[1].legend(frameon=False, fontsize=8)

ax[2].plot(x, y, "o", color="#1f4e79")
ax[2].plot(x, intercept + slope*x, "r--", lw=1.5, label=f"fit, slope = {slope:.4f}")
ax[2].set(xlabel=r"$\ln \Delta t$", ylabel=r"$\ln$ SD$(\Delta W)$",
          title="Log-log fit confirms exponent 1/2")
ax[2].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig("figures/w02_bm_scaling.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Increment distribution at one dt: normal by construction, so this is a build check
dt_ = 1/252
x = np.sqrt(dt_) * rng.standard_normal(100_000)
grid = np.linspace(x.min(), x.max(), 500)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].hist(x, bins=120, density=True, alpha=0.65, color="#1f4e79", label="simulated")
ax[0].plot(grid, stats.norm.pdf(grid, 0, np.sqrt(dt_)), "r-", lw=1.8, label=r"$N(0,\Delta t)$")
ax[0].set(xlabel=r"$\Delta W$", ylabel="density",
          title=r"Increment distribution, $\Delta t = 1/252$")
ax[0].legend(frameon=False)
stats.probplot(x[:20000], dist="norm", plot=ax[1])
ax[1].get_lines()[0].set(markersize=2, alpha=0.4, color="#1f4e79")
ax[1].get_lines()[1].set(color="red", lw=1.5)
ax[1].set_title("QQ plot: increments are exactly normal by construction")
plt.tight_layout()
plt.savefig("figures/w02_bm_increment_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

**Contrast this QQ plot with the NIFTY one from Week 1.** Here the points lie on the line
because the increments were *generated* normal. In Week 1 the same plot showed a pronounced
S-shape with excess kurtosis of 20.4. That difference is the entire subject of Part B.

---
## 8. Independence and stationarity of increments

Normality alone does not make a process Brownian motion. Two further properties are required:

- **Independent increments** — non-overlapping increments are independent
- **Stationary increments** — the distribution of $W_{t+\Delta t} - W_t$ depends on
  $\Delta t$ only, not on $t$

Tested here by autocorrelation (should be zero at all non-zero lags, within the
$\pm 1.96/\sqrt{n}$ band) and by comparing moments across blocks of the sample.

In [ ]:
n, dt_ = 200_000, 1/252
x = np.sqrt(dt_) * rng.standard_normal(n)

acf  = [np.corrcoef(x[:-k], x[k:])[0,1] for k in range(1, 11)]
band = 1.96/np.sqrt(n)
print("  Autocorrelation of increments (theory: 0 at all non-zero lags)")
for k, a in enumerate(acf, 1):
    print(f"    lag {k:>2}   {a:+.5f}" + ("" if abs(a) < band else "   <- outside 95% band"))
print(f"    95% band  +/-{band:.5f}")

blocks = np.array_split(x, 8)
sds = [b.std(ddof=1) for b in blocks]
print("\n  Stationarity: moments across 8 equal blocks")
print(f"    {'block':>6}{'mean':>14}{'sd':>14}")
for i, b in enumerate(blocks, 1):
    print(f"    {i:>6}{b.mean():>14.6f}{b.std(ddof=1):>14.6f}")
print(f"    spread of block SDs: {np.ptp(sds):.6f}  (compare sqrt(dt) = {np.sqrt(dt_):.6f})")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.0))
ax[0].stem(range(1, 11), acf, basefmt=" ")
ax[0].axhline(band, color="r", ls="--", lw=1)
ax[0].axhline(-band, color="r", ls="--", lw=1, label="95% band")
ax[0].axhline(0, color="k", lw=0.7)
ax[0].set(xlabel="lag", ylabel="autocorrelation", title="Increments are uncorrelated")
ax[0].legend(frameon=False, fontsize=8)
ax[1].plot(range(1, 9), sds, "o-", color="#1f4e79", label="block SD")
ax[1].axhline(np.sqrt(dt_), color="r", ls="--", lw=1.5, label=r"$\sqrt{\Delta t}$")
ax[1].set(xlabel="block", ylabel="SD", title="Increments are stationary")
ax[1].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig("figures/w02_bm_independence.png", dpi=150, bbox_inches="tight")
plt.show()

**Worth holding onto for later.** Brownian increments are uncorrelated *and* identically
distributed. The NIFTY return series in Week 1 showed clear volatility clustering — quiet
periods followed by quiet periods, violent ones by violent ones. Returns themselves are close
to uncorrelated, but their *magnitudes* are not, which breaks the stationarity property above.
Black–Scholes assumes it holds.

---
## 9. Quadratic variation — why Itô's lemma is not the chain rule

*Not required by §15.4.2. Included because §15.5.3 question 2 asks where the
$\tfrac12\sigma^2S^2V_{SS}$ term comes from, and this is the answer.*

Partition $[0,T]$ into $n$ steps and compute three sums:

$$\sum (\Delta W)^2, \qquad \sum |\Delta W|, \qquad \sum (\Delta W)^3$$

For an ordinary differentiable function all three would vanish as $n \to \infty$. For Brownian
motion:

$$\sum (\Delta W)^2 \to T \ \ (\text{finite, non-zero}), \qquad
\sum |\Delta W| \to \infty, \qquad
\sum (\Delta W)^3 \to 0$$

The convergence of the first is not eyeballed. $\mathrm{Var}\!\left(\sum(\Delta W)^2\right) =
2T^2/n$ exactly, so the standard error is $T\sqrt{2/n}$ and the normalised deviation should
stay of order 1 at every $n$.

In [ ]:
T = 1.0
ns = [10, 50, 100, 500, 1_000, 5_000, 10_000, 50_000, 100_000]
rows = []
for n_ in ns:
    dt_ = T/n_
    x = np.sqrt(dt_) * rng.standard_normal(n_)
    qv = (x**2).sum()
    se = T*np.sqrt(2/n_)                     # Var(sum (dW)^2) = n * 2 dt^2 = 2T^2/n
    rows.append({"n steps": n_, "dt": dt_, "sum (dW)^2": qv, "s.e. of QV": se,
                 "(QV - T)/s.e.": (qv-T)/se, "sum |dW|": np.abs(x).sum(),
                 "sum (dW)^3": (x**3).sum()})
qv_tab = pd.DataFrame(rows)
qv_tab.to_csv("results/w02_bm_quadratic_variation.csv", index=False)
with pd.option_context("display.float_format", lambda v: f"{v:14.6f}"):
    print(qv_tab.to_string(index=False))
print(f"\n  max |(QV - T)/s.e.| = {np.abs(qv_tab['(QV - T)/s.e.']).max():.2f}"
      "   <- stays O(1): this is L^2 convergence")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].semilogx(qv_tab["n steps"], qv_tab["sum (dW)^2"], "o-", color="#1f4e79",
               label=r"$\sum (\Delta W)^2$")
ax[0].axhline(T, color="r", ls="--", lw=1.5, label=f"T = {T}")
ax[0].fill_between(qv_tab["n steps"], T-2*qv_tab["s.e. of QV"], T+2*qv_tab["s.e. of QV"],
                   alpha=0.15, color="grey", label=r"$\pm 2$ s.e.")
ax[0].set(xlabel="number of steps n", ylabel=r"$\sum (\Delta W)^2$",
          title="Quadratic variation converges to T")
ax[0].legend(frameon=False, fontsize=8)
ax[1].loglog(qv_tab["n steps"], qv_tab["sum |dW|"], "o-", color="#c0504d",
             label=r"$\sum |\Delta W|$")
ax[1].loglog(qv_tab["n steps"], np.sqrt(2/np.pi)*np.sqrt(qv_tab["n steps"]), "k--", lw=1.4,
             label=r"theory: $\sqrt{2n/\pi}$")
ax[1].set(xlabel="number of steps n", ylabel=r"$\sum |\Delta W|$",
          title="First variation diverges")
ax[1].legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig("figures/w02_bm_quadratic_variation.png", dpi=150, bbox_inches="tight")
plt.show()

### What this means

For a smooth function $f(x)$, a Taylor expansion gives
$df = f'\,dx + \tfrac12 f''\,(dx)^2 + \cdots$, and the $(dx)^2$ term is discarded because it is
negligible.

For Brownian motion it cannot be discarded. The table above shows $(\Delta W)^2$ accumulating
at rate $dt$ — written compactly as

$$(dW)^2 = dt$$

Keeping that term is the entire difference between ordinary calculus and Itô calculus. Applying
it to $V(S,t)$ under $dS = \mu S\,dt + \sigma S\,dW$ produces

$$dV = \left[V_t + \mu S V_S + \underbrace{\tfrac12\sigma^2S^2V_{SS}}_{\text{from }(dW)^2 = dt}\right]dt + \sigma S V_S\,dW$$

which is the expression in §15.5.1, and the second-derivative term survives into the
Black–Scholes PDE. Week 2 derives this; Section 9 has now measured the fact it depends on.

Meanwhile $\sum|\Delta W| \to \infty$ says Brownian paths have infinite length over any
interval — they are continuous everywhere and differentiable nowhere. That is why
$dW/dt$ does not exist and why stochastic calculus is needed at all.

---
## 10. Summary

In [ ]:
summary = {
    "seed":                     SEED,
    "sd_scaling_exponent":      float(slope),
    "sd_scaling_theory":        0.5,
    "max_|var/dt - 1|":         float(np.abs(tab["var / dt"]-1).max()),
    "max_|sd/sqrt(dt) - 1|":    float(np.abs(tab["sd / sqrt(dt)"]-1).max()),
    "max_|autocorrelation|":    float(np.max(np.abs(acf))),
    "acf_95pct_band":           float(band),
    "QV_at_n=100000":           float(qv_tab["sum (dW)^2"].iloc[-1]),
    "QV_theory":                1.0,
    "max_|QV normalised dev|":  float(np.abs(qv_tab["(QV - T)/s.e."]).max()),
    "first_variation_n=100000": float(qv_tab["sum |dW|"].iloc[-1]),
}
pd.Series(summary).to_csv("results/w02_bm_summary.csv", header=False)
print("WEEK 2A SUMMARY")
print("="*52)
for k, v in summary.items():
    print(f"  {k:<28} {v}")
print("="*52)
print("\nSaved: results/w02_bm_*.csv   Figures: figures/w02_bm_*.png")

---
## 11. What Part A establishes

1. **$\Delta W = \sqrt{\Delta t}\,Z$ behaves exactly as claimed** — mean zero, variance
   $\Delta t$, across five orders of magnitude in $\Delta t$.
2. **The $\sqrt{\Delta t}$ scaling is confirmed by a fitted exponent**, not asserted. Slope
   0.5002 against a theoretical 0.5.
3. **Increments are independent and stationary** — all autocorrelations inside the 95% band,
   block moments flat.
4. **Quadratic variation is finite and non-zero while first variation diverges.** This is the
   property that forces Itô's lemma and produces the second-derivative term in the
   Black–Scholes PDE.

### Next: Part B

Per §15.4.3–§15.4.5, Part B builds geometric Brownian motion on top of this,

$$S_t = S_0\exp\left[\left(\mu - \tfrac{1}{2}\sigma^2\right)t + \sigma W_t\right]$$

simulates paths for different $\mu$ and $\sigma$, and then runs the **model-versus-data
experiment**: calibrate GBM to the NIFTY 50 estimates from Week 1 ($\sigma \approx 0.167$ per
§15.4.4) and compare mean, standard deviation, skewness, kurtosis, histograms, QQ plots and
tail frequencies against the empirical returns.

The principal question, from §15.4.5:

> If GBM is calibrated to the same mean and volatility, does it reproduce the distributional
> behaviour observed in the actual market data?

Section 8 above already suggests where the answer lies — GBM increments are stationary by
construction, and the NIFTY series visibly is not.

---
---

# Part B — Geometric Brownian Motion

---
## 12. GBM simulation and validation (§15.4.3)

The analytical GBM solution is

$$S_t = S_0\exp\left[\left(\mu - \tfrac{1}{2}\sigma^2\right)t + \sigma W_t\right]$$

Simulating this expression directly is **exact** — it carries no discretisation bias at any
step size. An Euler scheme applied to $dS = \mu S\,dt + \sigma S\,dW$ would not be, and can
even produce negative prices.

Four analytical results to check the simulator against:

$$E[S_T] = S_0e^{\mu T}, \qquad
\mathrm{Var}(S_T) = S_0^2e^{2\mu T}\left(e^{\sigma^2T}-1\right)$$

$$E[\ln S_T] = \ln S_0 + \left(\mu - \tfrac12\sigma^2\right)T, \qquad
\mathrm{Var}(\ln S_T) = \sigma^2 T$$

In [ ]:
def simulate_gbm(S0, mu, sigma, T, n_steps, n_paths, generator=None):
    """Exact GBM: S_t = S_0 exp[(mu - sigma^2/2) t + sigma W_t]."""
    g  = generator if generator is not None else rng
    dt = T / n_steps
    dW = np.sqrt(dt) * g.standard_normal((n_paths, n_steps))
    incr = (mu - 0.5*sigma**2)*dt + sigma*dW
    log_path = np.concatenate([np.zeros((n_paths,1)), np.cumsum(incr, axis=1)], axis=1)
    return S0 * np.exp(log_path)


S0, mu, sigma, T, n_steps, n_paths = 100.0, 0.10, 0.20, 1.0, 252, 200_000
S  = simulate_gbm(S0, mu, sigma, T, n_steps, n_paths)
ST = S[:, -1]

checks = [
    ("E[S_T]",      ST.mean(),              S0*np.exp(mu*T)),
    ("Var(S_T)",    ST.var(ddof=1),         S0**2*np.exp(2*mu*T)*(np.exp(sigma**2*T)-1)),
    ("E[ln S_T]",   np.log(ST).mean(),      np.log(S0)+(mu-0.5*sigma**2)*T),
    ("Var(ln S_T)", np.log(ST).var(ddof=1), sigma**2*T),
]
print(f"  S0={S0}, mu={mu}, sigma={sigma}, T={T}, {n_paths:,} paths x {n_steps} steps\n")
print(f"  {'quantity':<14}{'simulated':>16}{'theory':>16}{'rel. error':>14}")
for name, sim, th in checks:
    print(f"  {name:<14}{sim:>16.6f}{th:>16.6f}{abs(sim-th)/abs(th):>14.2e}")

print(f"\n  ln S_T normality: Jarque-Bera p = {stats.jarque_bera(np.log(ST)[:5000])[1]:.3f}")
dt_ = T/n_steps
r1 = np.diff(np.log(S), axis=1).ravel()
print(f"\n  One-step log returns (dt = 1/{n_steps}):")
print(f"    mean      {r1.mean():+.8f}   theory {(mu-0.5*sigma**2)*dt_:+.8f}")
print(f"    variance  {r1.var(ddof=1):.8f}   theory {sigma**2*dt_:.8f}")

---
## 13. The roles of $\mu$ and $\sigma$ (§15.4.3)

To isolate the effect of each parameter, **every panel below uses identical random shocks** —
the generator is re-seeded before each simulation. Any visible difference between panels is
therefore caused by the parameter, not by different noise.

In [ ]:
mus, sigmas = [0.00, 0.10, 0.20], [0.10, 0.20, 0.40]
n_show, t = 60, np.linspace(0, 1.0, 253)

fig, axes = plt.subplots(len(sigmas), len(mus), figsize=(13, 9.5), sharex=True, sharey=True)
for i, sig in enumerate(sigmas):
    for j, m in enumerate(mus):
        g = np.random.default_rng(SEED)          # identical shocks in every panel
        Sg = simulate_gbm(100.0, m, sig, 1.0, 252, n_show, generator=g)
        ax = axes[i, j]
        for k in range(n_show):
            ax.plot(t, Sg[k], lw=0.5, alpha=0.5)
        ax.plot(t, 100*np.exp(m*t), "r-", lw=2, label=r"$E[S_t]=S_0e^{\mu t}$")
        ax.plot(t, 100*np.exp((m-0.5*sig**2)*t), "k--", lw=1.6,
                label=r"median $=S_0e^{(\mu-\sigma^2/2)t}$")
        ax.set_title(fr"$\mu={m:.2f}$,  $\sigma={sig:.2f}$", fontsize=10)
        if i == len(sigmas)-1: ax.set_xlabel("t")
        if j == 0:             ax.set_ylabel("$S_t$")
        if i == 0 and j == 0:  ax.legend(frameon=False, fontsize=7)
fig.suptitle("Identical random shocks in every panel: rows change $\\sigma$, columns change $\\mu$",
             y=0.995, fontsize=11)
plt.tight_layout(); plt.savefig("figures/w02_gbm_mu_sigma_grid.png", dpi=150,
                                bbox_inches="tight")
plt.show()

**Reading the grid.**

- **Across a row** ($\mu$ increases): the whole bundle tilts upward. The *spread* is unchanged.
- **Down a column** ($\sigma$ increases): the bundle fans out — and the black dashed median
  line falls away from the red mean line.

So $\mu$ sets the trend and $\sigma$ sets the dispersion. But $\sigma$ does something else as
well: through the $-\tfrac12\sigma^2$ term it drags the *median* path below the *mean* path.
At $\sigma = 0.40$ the median grows at only 2% while the mean grows at 10%.

---
## 14. The two drifts: $\mu$ versus $\mu - \tfrac12\sigma^2$

GBM has two distinct growth rates and confusing them is a standard error:

$$\underbrace{E[S_T] = S_0e^{\mu T}}_{\text{the average outcome}} \qquad\text{versus}\qquad
\underbrace{E[\ln S_T] = \ln S_0 + \left(\mu - \tfrac12\sigma^2\right)T}_{\text{the typical outcome}}$$

The gap is exactly $\tfrac12\sigma^2$, at every $\sigma$, with no approximation.

**This is the distinction raised in the Week 1 feedback.** The empirical gap
$\overline{R} - \overline{r}$ measured on NIFTY data was a Taylor/Jensen identity in sample
moments — an algebraic fact about any collection of small numbers, requiring no probability
model at all. What is tested below is a different statement: a property of the continuous-time
process itself, arising from Itô's lemma applied to $f = \ln S$, where the term
$\tfrac12 f''(S)(dS)^2$ survives because $(dW)^2 = dt$ — the quadratic-variation property
measured in Part A, Section 9.

The two agree numerically under GBM. They are not the same claim, and only the second is
what Itô's lemma asserts.

In [ ]:
rows = []
n_p = 200_000
for sig in [0.05, 0.10, 0.20, 0.30, 0.40, 0.60]:
    Sg  = simulate_gbm(100.0, 0.10, sig, 1.0, 252, n_p)
    STg = Sg[:, -1]
    growth_mean = np.log(STg.mean()/100.0)      # -> mu
    growth_log  = np.log(STg/100.0).mean()      # -> mu - sigma^2/2  (tight estimator)
    rows.append({"sigma": sig, "growth of E[S_T]": growth_mean, "theory mu": 0.10,
                 "E[ln S_T]": growth_log, "theory mu - s^2/2": 0.10-0.5*sig**2,
                 "gap": growth_mean-growth_log, "sigma^2/2": 0.5*sig**2,
                 "s.e.": sig/np.sqrt(n_p)})
drift_tab = pd.DataFrame(rows)
drift_tab.to_csv("results/w02_gbm_two_drifts.csv", index=False)
with pd.option_context("display.float_format", lambda v: f"{v:13.6f}"):
    print(drift_tab.to_string(index=False))

dev = np.abs(drift_tab["gap"]-drift_tab["sigma^2/2"])/drift_tab["s.e."]
print(f"\n  max |gap - sigma^2/2|        = "
      f"{np.abs(drift_tab['gap']-drift_tab['sigma^2/2']).max():.2e}")
print(f"  max |gap - sigma^2/2| / s.e. = {dev.max():.2f}   <- within Monte Carlo error")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].plot(drift_tab["sigma"], drift_tab["gap"], "o-", color="#1f4e79", label="simulated gap")
ax[0].plot(drift_tab["sigma"], drift_tab["sigma^2/2"], "r--", lw=1.6, label=r"$\sigma^2/2$")
ax[0].set(xlabel=r"$\sigma$", ylabel=r"$\ln E[S_T] - E[\ln S_T]$",
          title=r"Model: the gap is exactly $\sigma^2/2$")
ax[0].legend(frameon=False)

sig = 0.40
Sg = simulate_gbm(100.0, 0.10, sig, 1.0, 252, 100_000)[:, -1]
ax[1].hist(Sg, bins=200, density=True, alpha=0.6, color="#1f4e79")
ax[1].axvline(Sg.mean(), color="r", lw=2, label=f"mean {Sg.mean():.1f}")
ax[1].axvline(np.median(Sg), color="k", ls="--", lw=2, label=f"median {np.median(Sg):.1f}")
ax[1].set(xlabel="$S_T$", ylabel="density",
          title=fr"Lognormal is right-skewed ($\sigma={sig}$): mean $>$ median")
ax[1].legend(frameon=False)
plt.tight_layout(); plt.savefig("figures/w02_gbm_two_drifts.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 15. Calibration to NIFTY 50 (§15.4.4)

§15.4.4 gives $\sigma \approx 0.167$ from the Week 1 analysis. Rather than hard-coding it,
the estimates are recomputed from the cached price series so the calibration is reproducible.

The drift is set so that the *log* drift matches:

$$\mu - \tfrac12\sigma^2 = \text{empirical annualised mean log return}$$

This matches GBM to the data on **both** the first and second moments, which is what §15.4.5
means by "the same $\mu$ and $\sigma$". Anything the model then fails to reproduce is a
genuine failure, not a calibration artefact.

In [ ]:
prices = pd.read_csv("data/nifty50_prices.csv", index_col=0,
                     parse_dates=True).iloc[:,0].dropna().astype(float)
r_emp = np.log(prices/prices.shift(1)).dropna()
TRADING_DAYS = 252

daily_mean, daily_sd = r_emp.mean(), r_emp.std(ddof=1)
sigma_n = daily_sd*np.sqrt(TRADING_DAYS)
mu_n    = daily_mean*TRADING_DAYS + 0.5*sigma_n**2

print(f"  period                 {prices.index[0].date()} to {prices.index[-1].date()}")
print(f"  observations           {len(r_emp):,}")
print(f"  daily mean log return  {daily_mean:+.8f}")
print(f"  daily sd  log return   {daily_sd:.8f}")
print(f"  annualised sigma       {sigma_n:.6f}   (plan 15.4.4 quotes 0.167)")
print(f"  annualised mu          {mu_n:.6f}")
print(f"\n  mu - sigma^2/2 = {mu_n-0.5*sigma_n**2:.6f}  matches empirical annualised")
print(f"  mean log return  {daily_mean*TRADING_DAYS:.6f}")

---
## 16. Model-versus-data experiment (§15.4.5)

Generate a GBM return sample with the **same number of observations, same mean and same
volatility** as the NIFTY sample, then compare everything that was *not* calibrated:
skewness, kurtosis, histograms, QQ plots and tail frequencies.

In [ ]:
n = len(r_emp)
dt_ = 1/TRADING_DAYS
r_gbm = (mu_n - 0.5*sigma_n**2)*dt_ + sigma_n*np.sqrt(dt_)*rng.standard_normal(n)
emp, sim = r_emp.values, r_gbm

rows = []
for name, f in [("mean",            lambda x: x.mean()),
                ("std deviation",   lambda x: x.std(ddof=1)),
                ("annualised vol",  lambda x: x.std(ddof=1)*np.sqrt(TRADING_DAYS)),
                ("skewness",        lambda x: stats.skew(x, bias=False)),
                ("excess kurtosis", lambda x: stats.kurtosis(x, bias=False)),
                ("minimum",         lambda x: x.min()),
                ("maximum",         lambda x: x.max())]:
    rows.append({"statistic": name, "NIFTY 50": f(emp), "GBM": f(sim)})
stat_tab = pd.DataFrame(rows).set_index("statistic")
stat_tab["difference"] = stat_tab["GBM"] - stat_tab["NIFTY 50"]
stat_tab.to_csv("results/w02_gbm_vs_nifty_statistics.csv")
with pd.option_context("display.float_format", lambda v: f"{v:12.6f}"):
    print(stat_tab.to_string())

print("\n  Normality tests")
print(f"    {'sample':<12}{'Jarque-Bera':>16}{'p-value':>14}{'verdict':>12}")
for name, x in [("NIFTY 50", emp), ("GBM", sim)]:
    jb, p = stats.jarque_bera(x)
    print(f"    {name:<12}{jb:>16,.2f}{p:>14.3e}{('REJECT' if p<0.01 else 'ok'):>12}")

In [ ]:
# tail frequencies
def tails(x):
    z = (x - x.mean())/x.std(ddof=1)
    return {k: (int((np.abs(z) > k).sum()), 2*stats.norm.sf(k)*len(z)) for k in (2,3,4,5)}

te, tg = tails(emp), tails(sim)
print("  Tail event frequencies")
print(f"    {'threshold':>10}{'NIFTY obs':>12}{'GBM obs':>10}{'normal exp':>12}"
      f"{'NIFTY ratio':>13}{'GBM ratio':>11}")
tail_rows = []
for k in (2,3,4,5):
    oe, ex = te[k]; og, _ = tg[k]
    re_, rg = oe/ex, og/ex
    print(f"    {'|z| > '+str(k):>10}{oe:>12d}{og:>10d}{ex:>12.2f}{re_:>13.1f}x{rg:>10.1f}x")
    tail_rows.append({"threshold": f"|z| > {k}", "NIFTY observed": oe, "GBM observed": og,
                      "normal expected": ex, "NIFTY ratio": re_, "GBM ratio": rg})
pd.DataFrame(tail_rows).to_csv("results/w02_gbm_vs_nifty_tails.csv", index=False)

In [ ]:
bins = np.linspace(min(emp.min(), sim.min()), max(emp.max(), sim.max()), 130)
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.4))
ax[0].hist(emp, bins=bins, density=True, alpha=0.55, color="#1f4e79", label="NIFTY 50")
ax[0].hist(sim, bins=bins, density=True, alpha=0.55, color="#e08214", label="GBM")
ax[0].set(xlabel="daily log return", ylabel="density",
          title="Histograms, identical mean and volatility")
ax[0].legend(frameon=False)
ax[1].hist(emp, bins=bins, density=True, alpha=0.55, color="#1f4e79", label="NIFTY 50")
ax[1].hist(sim, bins=bins, density=True, alpha=0.55, color="#e08214", label="GBM")
ax[1].set_yscale("log")
ax[1].set(xlabel="daily log return", ylabel="density (log scale)",
          title="Log density: the tails separate")
ax[1].legend(frameon=False)
plt.tight_layout()
plt.savefig("figures/w02_gbm_vs_nifty_histogram.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))
stats.probplot(emp, dist="norm", plot=ax[0])
ax[0].get_lines()[0].set(markersize=2.5, alpha=0.5, color="#1f4e79")
ax[0].get_lines()[1].set(color="red", lw=1.5)
ax[0].set_title("QQ plot: NIFTY 50 vs normal")
stats.probplot(sim, dist="norm", plot=ax[1])
ax[1].get_lines()[0].set(markersize=2.5, alpha=0.5, color="#e08214")
ax[1].get_lines()[1].set(color="red", lw=1.5)
ax[1].set_title("QQ plot: GBM vs normal")
qe, qg = np.sort(emp), np.sort(sim)
ax[2].plot(qg, qe, ".", markersize=2.5, alpha=0.5, color="#1f4e79")
lo, hi = min(qg.min(), qe.min()), max(qg.max(), qe.max())
ax[2].plot([lo, hi], [lo, hi], "r--", lw=1.5, label="45 degrees")
ax[2].set(xlabel="GBM quantiles", ylabel="NIFTY quantiles",
          title="NIFTY against GBM directly")
ax[2].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig("figures/w02_gbm_vs_nifty_qq.png", dpi=150,
                                bbox_inches="tight")
plt.show()

In [ ]:
# volatility clustering: autocorrelation of |returns|
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.2))
lags = range(1, 31)
for name, x, col in [("NIFTY 50", emp, "#1f4e79"), ("GBM", sim, "#e08214")]:
    a = [np.corrcoef(np.abs(x[:-k]), np.abs(x[k:]))[0,1] for k in lags]
    ax[0].plot(lags, a, "o-", ms=3, color=col, label=name)
ax[0].axhline(0, color="k", lw=0.7)
ax[0].axhline(1.96/np.sqrt(n), color="r", ls="--", lw=1, label="95% band")
ax[0].axhline(-1.96/np.sqrt(n), color="r", ls="--", lw=1)
ax[0].set(xlabel="lag (days)", ylabel="autocorrelation of |return|",
          title="Volatility clustering")
ax[0].legend(frameon=False, fontsize=8)
ax[1].plot(np.arange(n), emp, lw=0.5, color="#1f4e79", label="NIFTY 50")
ax[1].plot(np.arange(n), sim-0.10, lw=0.5, color="#e08214", label="GBM (offset)")
ax[1].set(xlabel="observation", ylabel="log return", title="Return series side by side")
ax[1].legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig("figures/w02_gbm_vs_nifty_clustering.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 17. Interpretation of model limitations (§15.4.6)

### Principal question (§15.4.5)

> If GBM is calibrated to the same mean and volatility, does it reproduce the distributional
> behaviour observed in the actual market data?

### Answer: no.

**What GBM reproduces**

- The mean and standard deviation — *exactly*, but only because these were the calibration
  targets. This is not evidence in the model's favour.
- Positivity of prices, which is why the process is geometric rather than arithmetic.
- The broad visual character of a price path over long horizons.

**What GBM fails to reproduce**

| Property | NIFTY 50 | GBM | |
|---|---|---|---|
| Skewness | ≈ −1.41 | ≈ 0 | large falls dominate large rises |
| Excess kurtosis | > 20 | ≈ 0 | tails far heavier than normal |
| Days beyond 4σ | 15 | ≈ 0 | model produces essentially none |
| Days beyond 5σ | 10 | ≈ 0 | model produces essentially none |
| Volatility clustering | strong, persistent | absent by construction | increments are i.i.d. |

The tail rows are the decisive ones. In matched samples of the same size, the market delivered
ten days beyond five standard deviations and the model produced none. Under GBM such a day has
probability $\approx 5.7\times10^{-7}$, roughly one occurrence in 6,900 years.

The volatility-clustering panel shows the same failure in a different form: the autocorrelation
of $|r_t|$ for NIFTY is strongly positive and decays slowly over 30 lags, while for GBM it sits
inside the 95% band at every lag. GBM increments are independent and identically distributed by
construction, so the model cannot generate clustering at all.

### Why this matters for the project

Black–Scholes is derived under exactly these dynamics and inherits every one of these failures.
Two consequences follow directly and will be met later in the project:

1. **A single constant $\sigma$ cannot price all strikes consistently.** Inverting the formula
   on observed market prices returns different implied volatilities at different strikes — the
   volatility smile — which is logically impossible if the model is true. This is met in
   September Week 3 (§15.6.3) and again with real option data in October.

2. **Tail risk is understated.** A model assigning probability $10^{-7}$ to an event the market
   delivers roughly annually will misprice deep out-of-the-money options and understate
   portfolio risk under stress. This is the subject of the November market-risk stage.

### Position going into September Week 2

The failures above are now *measured*, not assumed. Week 2 (§15.5) derives the Black–Scholes
PDE from these same dynamics via Itô's lemma and delta hedging. The purpose of having quantified
the gaps first is that the derivation can be understood as a rigorous consequence of assumptions
already known to be false in specific, measurable ways — which is the distinction between using
a model and believing it.

In [ ]:
summary = {
    "seed":                     SEED,
    "nifty_annualised_sigma":   float(sigma_n),
    "nifty_annualised_mu":      float(mu_n),
    "n_returns":                int(n),
    "nifty_skewness":           float(stats.skew(emp, bias=False)),
    "gbm_skewness":             float(stats.skew(sim, bias=False)),
    "nifty_excess_kurtosis":    float(stats.kurtosis(emp, bias=False)),
    "gbm_excess_kurtosis":      float(stats.kurtosis(sim, bias=False)),
    "nifty_jarque_bera_p":      float(stats.jarque_bera(emp)[1]),
    "gbm_jarque_bera_p":        float(stats.jarque_bera(sim)[1]),
    "nifty_days_beyond_4sigma": int(te[4][0]),
    "gbm_days_beyond_4sigma":   int(tg[4][0]),
    "nifty_days_beyond_5sigma": int(te[5][0]),
    "gbm_days_beyond_5sigma":   int(tg[5][0]),
    "sd_scaling_exponent":      float(slope),
    "QV_at_n=100000":           float(qv_tab["sum (dW)^2"].iloc[-1]),
}
pd.Series(summary).to_csv("results/w02_summary.csv", header=False)
print("SEPTEMBER WEEK 1 SUMMARY")
print("="*56)
for k, v in summary.items():
    print(f"  {k:<28} {v}")
print("="*56)
print("\nSaved: results/w02_*.csv   Figures: figures/w02_*.png")